<a href="https://colab.research.google.com/github/grasht/projects_ML_HW_5/blob/main/HW_5_tsk_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3 NLP and Attention Mechanism

# Part 1 Scaled Dot-Product Attention

In [66]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [67]:
import numpy as np
import pandas as pd

#Scaled Dot Product Attention
def sdp_attention(query, key, value):
  def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

  QK = query @ key.T
  dk = key.shape[1]
  scaledQK = QK/np.sqrt(dk)
  softmaxQK = softmax(scaledQK)
  return softmaxQK @ value, softmaxQK



# Part 2 Encoder-Decoder Model with Integrated Attention Mechanism



In [68]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)

        #I tried this with LSTM first, but when I realized we are using a small-scale
        #dataset in Part3 I switched to GRU
        #self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)

        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)

        # outputs: (batch, seq_len, hidden_dim)
        # hidden: (1, batch, hidden_dim)
        outputs, hidden = self.rnn(embedded)

        return outputs, hidden

In [69]:
class Attention(nn.Module):
  def __init__(self):
    super().__init__()

  #Copy this into the Class definition to avoid relying on a global function
  # def sdp_attention(query, key, value):
  #   def softmax(x):
  #     e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
  #     return e_x / np.sum(e_x, axis=-1, keepdims=True)

  #   QK = query @ key.T
  #   dk = key.shape[1]
  #   scaledQK = QK/np.sqrt(dk)
  #   softmaxQK = softmax(scaledQK)
  #   return softmaxQK @ value, softmaxQK

  #Need to reimplement sdp_attention using pytorch or it wont work with tensors
  def sdp_attention(self, query, key, value):
    dk = key.size(-1)
    scores = torch.bmm(query, key.transpose(1, 2)) / torch.sqrt(torch.tensor(dk, dtype=torch.float, device=key.device))
    attn_weights = torch.softmax(scores, dim=-1)
    context = torch.bmm(attn_weights, value)
    return context, attn_weights

  def forward(self, decoder_hidden, encoder_outputs):
    query = decoder_hidden[-1].unsqueeze(1)
    key = encoder_outputs
    value = encoder_outputs

    context, attn_weights = self.sdp_attention(query, key, value)

    return context, attn_weights

In [70]:
class Decoder(nn.Module):
  def __init__(self, output_dim, emb_dim, hid_dim):
    super().__init__()

    self.embedding = nn.Embedding(output_dim, emb_dim)
    #self.rnn = nn.LSTM(emb_dim + hid_dim, hid_dim, batch_first=True)
    self.rnn = nn.GRU(emb_dim + hid_dim, hid_dim, batch_first=True)

    self.fc = nn.Linear(hid_dim *2, output_dim)

    self.attention = Attention()

  def forward(self, input, hidden, encoder_outputs):
    input = input.unsqueeze(1)
    embedded = self.embedding(input)

    context, attn_weights = self.attention(hidden, encoder_outputs)
    rnn_input = torch.cat((embedded, context), dim=2)

    output, hidden = self.rnn(rnn_input, hidden)

    output = output.squeeze(1)
    context = context.squeeze(1)

    prediction = self.fc(torch.cat((output, context), dim=1))

    return prediction, hidden, attn_weights

In [71]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        trg_vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size, device=src.device)

        encoder_outputs, hidden = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, _ = self.decoder(input, hidden, encoder_outputs)
            outputs[:, t] = output

            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)

            input = trg[:, t] if teacher_force else top1

        return outputs

# Part 3 Pick a Data Set and Train the Model

In [72]:
# Load dataset
from datasets import load_dataset

dataset = load_dataset("bentrevett/multi30k")

def tokenize_de(text):
    return text.lower().split()

def tokenize_en(text):
    return text.lower().split()

In [73]:
def train(model, dataloader, optimizer, criterion, clip, device):
    model.train()
    epoch_loss = 0

    for src, trg in dataloader:
        src = src.to(device)   # German
        trg = trg.to(device)   # English

        optimizer.zero_grad()

        # output: [trg_len, batch_size, output_dim]
        output = model(src, trg, teacher_forcing_ratio=0.5)

        # Ignore <sos>
        output_dim = output.shape[-1]
        output = output[1:].reshape(-1, output_dim)
        trg = trg[1:].reshape(-1)

        # Compute loss
        loss = criterion(output, trg)

        # Backprop
        loss.backward()

        # Gradient clipping (important for RNNs)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)

In [74]:
def evaluate(model, dataloader, criterion, device):

    model.eval()
    epoch_loss = 0

    with torch.no_grad():

        for src, trg in dataloader:

            src = src.to(device)
            trg = trg.to(device)

            # No teacher forcing
            output = model(src, trg, teacher_forcing_ratio=0)

            output_dim = output.shape[-1]

            output = output[1:].view(-1, output_dim)
            trg = trg[1:].reshape(-1)

            loss = criterion(output, trg)

            epoch_loss += loss.item()

    return epoch_loss / len(dataloader)

In [75]:
from collections import Counter

def build_vocab(sentences, tokenizer, min_freq=2):
    counter = Counter()
    for sentence in sentences:
        counter.update(tokenizer(sentence))

    vocab = {
        "<pad>": 0,
        "<sos>": 1,
        "<eos>": 2,
        "<unk>": 3
    }

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

vocab_de = build_vocab(dataset["train"]["de"], tokenize_de)
vocab_en = build_vocab(dataset["train"]["en"], tokenize_en)

In [76]:
import torch
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    src_batch = []
    trg_batch = []

    for src, trg in batch:
        src_tensor = torch.tensor(process(src, vocab_de, tokenize_de), dtype=torch.long)
        trg_tensor = torch.tensor(process(trg, vocab_en, tokenize_en), dtype=torch.long)

        src_batch.append(src_tensor)
        trg_batch.append(trg_tensor)

    # Pad sequences
    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    trg_batch = pad_sequence(trg_batch, batch_first=True, padding_value=PAD_IDX)

    return src_batch, trg_batch

In [77]:
from torch.utils.data import Dataset

class TranslationDataset(Dataset):
    def __init__(self, hf_dataset, src_lang="de", trg_lang="en"):
        self.src_data = hf_dataset[src_lang]
        self.trg_data = hf_dataset[trg_lang]

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return self.src_data[idx], self.trg_data[idx]

In [78]:
from torch.utils.data import DataLoader

train_dataset = TranslationDataset(dataset["train"])
valid_dataset = TranslationDataset(dataset["validation"])
test_dataset  = TranslationDataset(dataset["test"])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [79]:
PAD_IDX = vocab_en["<pad>"]

def numericalize(sentence, vocab, tokenizer):
    tokens = tokenizer(sentence)
    return [vocab.get(token, vocab["<unk>"]) for token in tokens]

def process(sentence, vocab, tokenizer):
    return [vocab["<sos>"]] + \
           numericalize(sentence, vocab, tokenizer) + \
           [vocab["<eos>"]]

for batch in train_loader:
    src, trg = batch
    src = src.to(device)
    trg = trg.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
models2s = Seq2Seq(Encoder(len(vocab_de), 256, 512), Decoder(len(vocab_en), 256, 512))
models2s = models2s.to(device)
optimizer = torch.optim.Adam(models2s.parameters())

N_EPOCHS = 10
CLIP = 1.0

for epoch in range(N_EPOCHS):

    train_loss = train(models2s, train_loader, optimizer, criterion, CLIP, device)
    valid_loss = evaluate(models2s, valid_loader, criterion, device)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.3f}")
    print(f"Val Loss:   {valid_loss:.3f}")

Epoch 1
Train Loss: 4.306
Val Loss:   3.987
Epoch 2
Train Loss: 3.208
Val Loss:   3.860
Epoch 3
Train Loss: 2.639
Val Loss:   3.858
Epoch 4
Train Loss: 2.235
Val Loss:   3.940
Epoch 5
Train Loss: 1.982
Val Loss:   4.062


```
Epoch 1
Train Loss: 4.288
Val Loss:   4.018
Epoch 2
Train Loss: 3.183
Val Loss:   3.859
Epoch 3
Train Loss: 2.612
Val Loss:   3.897
Epoch 4
Train Loss: 2.224
Val Loss:   3.994
Epoch 5
Train Loss: 1.985
Val Loss:   4.093
Epoch 6
Train Loss: 1.799
Val Loss:   4.222
Epoch 7
Train Loss: 1.636
Val Loss:   4.341
Epoch 8
Train Loss: 1.503
Val Loss:   4.534
Epoch 9
Train Loss: 1.406
Val Loss:   4.696
Epoch 10
Train Loss: 1.302
Val Loss:   4.819```



In [ ]:
def translate_seq2seq(model, src, max_len, sos_idx, eos_idx, device):
    model.eval()

    src = src.unsqueeze(0).to(device)  # (1, src_len)

    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src)

    input_token = torch.tensor([sos_idx], device=device)

    outputs = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, _ = model.decoder(input_token, hidden, encoder_outputs)

        pred_token = output.argmax(1).item()
        outputs.append(pred_token)

        if pred_token == eos_idx:
            break

        input_token = torch.tensor([pred_token], device=device)

    return outputs

# Part 4 Simplified Transformer Model

In [ ]:
%pip install tokenizers

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, emb_dim, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, emb_dim)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, emb_dim, 2) * (-math.log(10000.0) / emb_dim)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)  # (1, max_len, emb_dim)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

In [ ]:
def sdp_attention_fn(Q, K, V, mask=None):
    dk = Q.size(-1)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(dk)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    attn = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn, V)

    return output, attn

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, n_heads):
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = emb_dim // n_heads

        self.q_linear = nn.Linear(emb_dim, emb_dim)
        self.k_linear = nn.Linear(emb_dim, emb_dim)
        self.v_linear = nn.Linear(emb_dim, emb_dim)

        self.fc_out = nn.Linear(emb_dim, emb_dim)

    def forward(self, x, mask=None):
        B, T, E = x.shape

        Q = self.q_linear(x)
        K = self.k_linear(x)
        V = self.v_linear(x)

        # Split heads
        Q = Q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        out, attn = sdp_attention_fn(Q, K, V, mask)

        # Concatenate heads
        out = out.transpose(1, 2).contiguous().view(B, T, E)

        return self.fc_out(out)

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim, ff_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, emb_dim)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, emb_dim, n_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attn = MultiHeadAttention(emb_dim, n_heads)
        self.ff = FeedForward(emb_dim, ff_dim)

        self.norm1 = nn.LayerNorm(emb_dim)
        self.norm2 = nn.LayerNorm(emb_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out = self.attn(x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))

        return x

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, emb_dim, n_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.self_attn = MultiHeadAttention(emb_dim, n_heads)
        self.cross_attn = MultiHeadAttention(emb_dim, n_heads)

        self.ff = FeedForward(emb_dim, ff_dim)

        self.norm1 = nn.LayerNorm(emb_dim)
        self.norm2 = nn.LayerNorm(emb_dim)
        self.norm3 = nn.LayerNorm(emb_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask=None, trg_mask=None):
        # Masked self-attention
        x = self.norm1(x + self.dropout(self.self_attn(x, trg_mask)))

        # Cross-attention
        x = self.norm2(x + self.dropout(self.cross_attn(x, src_mask)))

        # Feedforward
        x = self.norm3(x + self.dropout(self.ff(x)))

        return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, trg_vocab, emb_dim=64, n_heads=2, ff_dim=128, n_layers=2):
        super().__init__()

        self.src_emb = nn.Embedding(src_vocab, emb_dim)
        self.trg_emb = nn.Embedding(trg_vocab, emb_dim)

        self.pos_enc = PositionalEncoding(emb_dim)

        self.encoder = nn.ModuleList([
            EncoderLayer(emb_dim, n_heads, ff_dim) for _ in range(n_layers)
        ])

        self.decoder = nn.ModuleList([
            DecoderLayer(emb_dim, n_heads, ff_dim) for _ in range(n_layers)
        ])

        self.fc_out = nn.Linear(emb_dim, trg_vocab)

    def forward(self, src, trg):
        src = self.pos_enc(self.src_emb(src))
        trg = self.pos_enc(self.trg_emb(trg))

        for layer in self.encoder:
            src = layer(src)

        for layer in self.decoder:
            trg = layer(trg, src)

        return self.fc_out(trg)

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# Initialize BPE tokenizer
tokenizer = Tokenizer(models.BPE())

# Basic pre-tokenization (whitespace)
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.BpeTrainer(
    vocab_size=10000,
    special_tokens=["<pad>", "<sos>", "<eos>", "<unk>"]
)

# Combine German + English text
def sentence_iterator():
    for item in dataset["train"]:
        yield item["de"]
        yield item["en"]

tokenizer.train_from_iterator(sentence_iterator(), trainer)

In [ ]:
# def create_src_mask(src, pad_idx):
#     # src: (batch, src_len)
#     mask = (src != pad_idx).unsqueeze(1).unsqueeze(2)
#     # (batch, 1, 1, src_len)
#     return mask


# def create_trg_mask(trg, pad_idx):
#     batch_size, trg_len = trg.shape

#     # Padding mask
#     trg_pad_mask = (trg != pad_idx).unsqueeze(1).unsqueeze(2)
#     # (batch, 1, 1, trg_len)

#     # Causal mask
#     trg_sub_mask = torch.tril(torch.ones((trg_len, trg_len), device=trg.device)).bool()
#     # (trg_len, trg_len)

#     # Combine
#     trg_mask = trg_pad_mask & trg_sub_mask
#     # (batch, 1, trg_len, trg_len)

#     return trg_mask

In [ ]:
def encode(sentence):
    tokens = tokenizer.encode(sentence)
    return tokens.ids

In [ ]:
SOS_IDX = tokenizer.token_to_id("<sos>")
EOS_IDX = tokenizer.token_to_id("<eos>")
PAD_IDX = tokenizer.token_to_id("<pad>")

def process(sentence):
    return [SOS_IDX] + encode(sentence) + [EOS_IDX]

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    src_batch = []
    trg_batch = []

    for src, trg in batch:
        src_tensor = torch.tensor(process(src), dtype=torch.long)
        trg_tensor = torch.tensor(process(trg), dtype=torch.long)

        src_batch.append(src_tensor)
        trg_batch.append(trg_tensor)

    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    trg_batch = pad_sequence(trg_batch, batch_first=True, padding_value=PAD_IDX)

    return src_batch, trg_batch

In [ ]:
def decode(ids):
    return tokenizer.decode(ids, skip_special_tokens=True)

In [ ]:
vocab_size = tokenizer.get_vocab_size()

# model = Transformer(
#     len(vocab_de),
#     len(vocab_en),
#     pad_idx=PAD_IDX
# ).to(device)

model = Transformer(
    vocab_size,   # src
    vocab_size   # trg
).to(device)

In [ ]:
print(trg.shape)
print(trg)
output = model(src, trg[:, :-1])

output_dim = output.shape[-1]

output = output.contiguous().view(-1, output_dim)
trg = trg[:, 1:].contiguous().view(-1)

loss = criterion(output, trg)

In [ ]:
def translate_transformer(model, src, max_len, sos_idx, eos_idx, device):
    model.eval()

    src = src.unsqueeze(0).to(device)

    trg_tokens = [sos_idx]

    for _ in range(max_len):
        trg_tensor = torch.tensor(trg_tokens, dtype=torch.long, device=device).unsqueeze(0)

        with torch.no_grad():
            output = model(src, trg_tensor)

        pred_token = output[:, -1, :].argmax(1).item()
        trg_tokens.append(pred_token)

        if pred_token == eos_idx:
            break

    return trg_tokens[1:]

In [ ]:
def decode_tokens(token_ids, tokenizer):
    return tokenizer.decode(token_ids, skip_special_tokens=True)

In [ ]:
%pip install sacrebleu

In [ ]:
import sacrebleu

def compute_bleu(model, dataloader, tokenizer, sos_idx, eos_idx, device, model_type="seq2seq"):
    references = []
    hypotheses = []

    for src_batch, trg_batch in dataloader:
        for i in range(src_batch.size(0)):

            src = src_batch[i]
            trg = trg_batch[i]

            # Remove padding
            src = src[src != PAD_IDX]
            trg = trg[trg != PAD_IDX]

            # Generate prediction
            if model_type == "seq2seq":
                pred_tokens = translate_seq2seq(model, src, max_len=50, sos_idx=sos_idx, eos_idx=eos_idx, device=device)
            else:
                pred_tokens = translate_transformer(model, src, max_len=50, sos_idx=sos_idx, eos_idx=eos_idx, device=device)

            # Decode
            pred_sentence = decode_tokens(pred_tokens, tokenizer)
            trg_sentence = decode_tokens(trg.tolist(), tokenizer)

            hypotheses.append(pred_sentence)
            references.append([trg_sentence])  # sacrebleu expects list of refs

    bleu = sacrebleu.corpus_bleu(hypotheses, references)

    return bleu.score

#Results and Blue Scores

In [ ]:
bleu_seq2seq = compute_bleu(models2s, test_loader, tokenizer, SOS_IDX, EOS_IDX, device, "seq2seq")
print("Seq2Seq BLEU:", bleu_seq2seq)

bleu_transformer = compute_bleu(model, test_loader, tokenizer, SOS_IDX, EOS_IDX, device, "transformer")
print("Transformer BLEU:", bleu_transformer)